# LightRAG 10-K Test Pipeline - HON First

**Goal**: Test streamlined LightRAG indexing with full metadata preservation

**Test Strategy**:
1. Index HON (Honeywell) document only
2. Query with 2 test questions from finder_train.parquet
3. Verify metadata retrieval works
4. If successful, batch index remaining priority tickers

**Models**:
- Embedding: BAAI/bge-large-en-v1.5 (GPU)
- LLM: Mistral-8b (API)
- Reranker: BAAI/bge-reranker-base (GPU)
- Chunking: 1200 tokens, 100 overlap

**Priority Tickers**: HON, MSI, MRO, CE, HAS, VRSK, BIIB, ROP, CTAS, AMP

In [ ]:
# Standard library
import os
import json
import asyncio
from pathlib import Path
from functools import partial
from datetime import datetime
import re

# Third-party
from dotenv import load_dotenv
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import nest_asyncio

# LightRAG
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

# ML Models
from sentence_transformers import SentenceTransformer, CrossEncoder

# Enable nested async for Jupyter
nest_asyncio.apply()
load_dotenv()

# ============================================
# Configuration
# ============================================

# Paths
PARSED_DOCS_PATH = "parsed_10k_documents.json"
WORKING_DIR = "./lightrag_test_workspace"
TEST_QUESTIONS_PATH = "finder_train.parquet"
RESULTS_PATH = "test_results_hon.json"

# Model names
BGE_MODEL_NAME = "BAAI/bge-large-en-v1.5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
MISTRAL_MODEL = "ministral-8b-latest"
EMBEDDING_DIM = 1024

# GPU setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"🚀 Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*60 + "\n")

# Priority tickers
PRIORITY_TICKERS = ["HON", "MSI", "MRO", "CE", "HAS", "VRSK", "BIIB", "ROP", "CTAS", "AMP"]

print("✓ Configuration loaded")

In [ ]:
print("Loading models...\n")

# ============================================
# 1. BGE Embedding Model (GPU)
# ============================================
print(f"Loading {BGE_MODEL_NAME} on {DEVICE}...")
bge_model = SentenceTransformer(BGE_MODEL_NAME, device=DEVICE)
print(f"✓ Embedding model loaded")
print(f"  Dimension: {EMBEDDING_DIM}")
print(f"  Device: {bge_model.device}\n")

# Test embedding
test_embed = bge_model.encode(["test"], normalize_embeddings=True)
assert test_embed.shape[1] == EMBEDDING_DIM
print(f"  Test embedding: {test_embed.shape} ✓\n")

# ============================================
# 2. Mistral LLM (API)
# ============================================
async def mistral_llm_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    """Mistral LLM wrapper"""
    return await openai_complete_if_cache(
        MISTRAL_MODEL,
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("MISTRAL_API_KEY"),
        base_url="https://api.mistral.ai/v1",
        **kwargs
    )

print(f"✓ LLM configured: {MISTRAL_MODEL}\n")

# ============================================
# 3. BGE Embedding Function (Async Wrapper)
# ============================================
async def bge_embedding_func(texts: list[str]) -> np.ndarray:
    """Async wrapper for BGE embeddings"""
    return bge_model.encode(texts, normalize_embeddings=True, device=DEVICE)

print("✓ Embedding function wrapped\n")

# ============================================
# 4. HuggingFace Cross-Encoder Reranker (GPU)
# ============================================
print(f"Loading {RERANKER_MODEL_NAME} on {DEVICE}...")
reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE, max_length=512)
print(f"✓ Reranker loaded on {reranker.model.device}\n")

async def hf_rerank_func(query: str, documents: list[str], top_n: int = 20, **kwargs):
    """HuggingFace CrossEncoder reranker with GPU acceleration"""
    # Create query-document pairs
    pairs = [[query, doc] for doc in documents]
    
    # Predict relevance scores (on GPU)
    scores = reranker.predict(
        pairs,
        convert_to_tensor=True,
        show_progress_bar=False,
        batch_size=32  # GPU batch processing
    )
    
    # Format results
    results = [
        {"index": i, "relevance_score": float(score)}
        for i, score in enumerate(scores)
    ]
    
    # Sort by relevance and return top_n
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:top_n]

print("="*60)
print("✓ All models loaded successfully")
print("="*60)

In [ ]:
print("\nInitializing LightRAG...\n")

# Create workspace
Path(WORKING_DIR).mkdir(exist_ok=True)

# ============================================
# Initialize LightRAG Instance
# ============================================
rag = LightRAG(
    working_dir=WORKING_DIR,
    
    # LLM
    llm_model_func=mistral_llm_func,
    llm_model_name=MISTRAL_MODEL,
    
    # Embeddings
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBEDDING_DIM,
        max_token_size=8192,
        func=bge_embedding_func
    ),
    
    # Chunking (LightRAG handles this)
    chunk_token_size=1200,
    chunk_overlap_token_size=100,
    
    # Reranker (GPU-accelerated)
    rerank_model_func=hf_rerank_func
)

await rag.initialize_storages()

print("✓ LightRAG initialized")
print(f"  Working dir: {WORKING_DIR}")
print(f"  Chunk size: 1200 tokens, 100 overlap")
print(f"  Reranker: GPU-accelerated\n")

# ============================================
# Load Documents
# ============================================
with open(PARSED_DOCS_PATH, 'r') as f:
    documents = json.load(f)

print(f"✓ Loaded {len(documents)} company documents\n")

# ============================================
# Index Function with Metadata
# ============================================
async def index_document(ticker: str, doc_data: dict):
    """Index document with full metadata encoded in file_path"""
    
    # Encode metadata as compact JSON
    metadata_json = json.dumps({
        "ticker": ticker,
        "company": doc_data.get("company_name", ""),
        "period": doc_data.get("period_end_date", ""),
        "source": doc_data.get("source_file", "")
    }, separators=(',', ':'))
    
    # Insert document (LightRAG chunks automatically)
    await rag.ainsert(
        input=doc_data["text"],
        ids=ticker,
        file_paths=metadata_json
    )
'''
# ============================================
# TEST: Index HON Only
# ============================================
print("="*60)
print("📍 TEST: Indexing HON (Honeywell) only")
print("="*60)

if "HON" not in documents:
    print("❌ ERROR: HON not found in documents!")
else:
    print(f"\nCompany: {documents['HON']['company_name']}")
    print(f"Period: {documents['HON']['period_end_date']}")
    print(f"Text length: {len(documents['HON']['text']):,} chars\n")
    
    print("Indexing HON... (this may take a few minutes)")
    await index_document("HON", documents["HON"])
    
    print("\n✓ HON indexed successfully!")
    print("  - LightRAG chunked document automatically")
    print("  - Built knowledge graph entities/relations")
    print("  - Stored metadata for citation/provenance")

print("\n" + "="*60)
print("Ready for querying!")
print("="*60)
'''
# After successful test, uncomment to index remaining tickers:
# 
print("\n\n📍 Indexing remaining priority tickers...")
BATCH_PRIORITY = [t for t in PRIORITY_TICKERS if t != "HON" and t in documents]
for ticker in tqdm(BATCH_PRIORITY, desc="Priority batch"):
    await index_document(ticker, documents[ticker])

print(f"\n✓ Indexed {len(BATCH_PRIORITY)} priority tickers")

In [ ]:
print("\nLoading test questions...\n")

# ============================================
# Load Test Questions
# ============================================
df = pd.read_parquet(TEST_QUESTIONS_PATH)

# Filter for HON questions (word boundary search)
def has_hon(refs):
    refs_str = ' '.join(str(r) for r in refs) if isinstance(refs, (list, tuple)) else str(refs)
    return bool(re.search(r'\bHON\b', refs_str) or 'Honeywell' in refs_str)

hon_mask = df['references'].apply(has_hon)
hon_questions = df[hon_mask].head(2)

print(f"Found {hon_mask.sum()} total HON questions")
print(f"Testing with {len(hon_questions)} questions\n")

# Display test questions
for i, (idx, row) in enumerate(hon_questions.iterrows(), 1):
    print(f"Question {i}:")
    print(f"  ID: {row['_id']}")
    print(f"  Text: {row['text']}")
    print(f"  Type: {row['type']} | Category: {row['category']}")
    print(f"  Expected answer: {row['answer'][:100]}...\n")

# ============================================
# Query Function
# ============================================
async def query_and_extract(question_row):
    """Query LightRAG and extract metadata from results"""
    
    question_id = question_row["_id"]
    question_text = question_row["text"]
    
    print(f"\n{'='*60}")
    print(f"🔍 Querying: {question_text}")
    print("="*60)
    
    # Query with hybrid mode + reranking
    context = await rag.aquery(
        question_text,
        param=QueryParam(
            mode="hybrid",  # Use KG + vector search
            top_k=60,       # Entities/relations
            chunk_top_k=20, # Text chunks
            only_need_context=True,  # Just context for testing
            enable_rerank=True  # GPU reranker
        )
    )
    
    print(f"\n✓ Retrieved context ({len(context)} chars)")
    
    # Parse context to extract citations
    # LightRAG context includes file_path with our metadata JSON
    # (Actual parsing would extract and decode the JSON)
    
    return {
        "question_id": question_id,
        "question": question_text,
        "expected_answer": question_row["answer"],
        "expected_type": question_row["type"],
        "category": question_row["category"],
        "retrieved_context": context,
        "context_length": len(context),
        "timestamp": datetime.now().isoformat()
    }

# ============================================
# Run Test Queries
# ============================================
print("\n" + "="*60)
print("🎯 Running test queries")
print("="*60)

test_results = []
for idx, row in hon_questions.iterrows():
    result = await query_and_extract(row)
    test_results.append(result)
    
    # Preview context
    print(f"\nContext preview:")
    print(result["retrieved_context"][:500] + "...\n")

# ============================================
# Save Results
# ============================================
with open(RESULTS_PATH, 'w') as f:
    json.dump(test_results, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print(f"✓ Test complete!")
print("="*60)
print(f"\nResults saved to: {RESULTS_PATH}")
print(f"Total questions tested: {len(test_results)}")
print(f"\nNext steps:")
print("  1. Verify metadata appears in context")
print("  2. Check citation/provenance quality")
print("  3. If successful, uncomment batch indexing in Cell 4")
print("  4. Run evaluation metrics")

In [ ]:
print("\n" + "="*60)
print("🎯 Query ALL Priority Ticker Questions")
print("="*60)
print()

# ============================================
# Helper: Convert numpy/pandas types to Python types
# ============================================

def to_python_type(val):
    """Convert pandas/numpy types to native Python types for JSON serialization"""
    if pd.isna(val):
        return None
    elif isinstance(val, (np.integer, np.int64, np.int32)):
        return int(val)
    elif isinstance(val, (np.floating, np.float64, np.float32)):
        return float(val)
    elif isinstance(val, np.bool_):
        return bool(val)
    elif isinstance(val, np.ndarray):
        return val.tolist()
    elif isinstance(val, (list, tuple)):
        return str(val)  # Convert lists to string for JSON
    else:
        return val

# ============================================
# Filter Questions for Priority Tickers
# ============================================

def extract_tickers_from_text(text: str) -> list:
    """Extract ticker symbols from text (same logic as count_ticker_occurrences.py)"""
    tickers = []
    
    # Pattern 1: Ticker in parentheses
    paren_tickers = re.findall(r'\(([A-Z]{1,5})\)', text)
    tickers.extend(paren_tickers)
    
    # Pattern 2: Ticker at end with dash or comma
    end_tickers = re.findall(r'[,\-]\s*([A-Z]{1,5})\s*[,.]?\s*$', text)
    tickers.extend(end_tickers)
    
    return list(set(t for t in tickers if len(t) <= 5))

def filter_priority_questions(df: pd.DataFrame) -> pd.DataFrame:
    """Filter questions for priority tickers"""
    
    def has_priority_ticker(text):
        tickers = extract_tickers_from_text(text)
        return any(ticker in PRIORITY_TICKERS for ticker in tickers)
    
    mask = df['text'].apply(has_priority_ticker)
    return df[mask]

# Load full dataset
print("Loading all questions...")
df_all = pd.read_parquet(TEST_QUESTIONS_PATH)
print(f"✓ Loaded {len(df_all)} total questions\n")

# Filter for priority tickers
print("Filtering for priority tickers...")
priority_questions = filter_priority_questions(df_all)
print(f"✓ Found {len(priority_questions)} questions for priority tickers\n")

# Count per ticker
print("Questions per ticker:")
ticker_counts = {}
for ticker in PRIORITY_TICKERS:
    count = priority_questions['text'].apply(
        lambda text: ticker in extract_tickers_from_text(text)
    ).sum()
    ticker_counts[ticker] = count
    print(f"  {ticker}: {count} questions")

print(f"\n  TOTAL: {len(priority_questions)} questions")
print()

# ============================================
# Query Function (with error handling & type conversion)
# ============================================

async def query_with_metadata(question_row, idx, total):
    """Query LightRAG and extract metadata from results"""
    
    question_id = str(question_row["_id"])
    question_text = str(question_row["text"])
    
    print(f"[{idx}/{total}] {question_text[:70]}...")
    
    try:
        # Query with hybrid mode + reranking
        context = await rag.aquery(
            question_text,
            param=QueryParam(
                mode="mix",
                top_k=60,
                chunk_top_k=20,
                only_need_context=True,
                enable_rerank=True
            )
        )
        
        print(f"         ✓ Retrieved {len(context):,} chars")
        
        return {
            "question_id": question_id,
            "question": question_text,
            "expected_answer": to_python_type(question_row["answer"]),
            "expected_type": to_python_type(question_row.get("type")),
            "category": to_python_type(question_row.get("category")),
            "reasoning": to_python_type(question_row.get("reasoning")),
            "references": to_python_type(question_row.get("references")),
            "retrieved_context": context,
            "context_length": len(context),
            "timestamp": datetime.now().isoformat(),
            "status": "success"
        }
    
    except Exception as e:
        print(f"         ❌ Error: {str(e)}")
        return {
            "question_id": question_id,
            "question": question_text,
            "expected_answer": to_python_type(question_row["answer"]),
            "expected_type": to_python_type(question_row.get("type")),
            "category": to_python_type(question_row.get("category")),
            "error": str(e),
            "timestamp": datetime.now().isoformat(),
            "status": "error"
        }

# ============================================
# Run Queries for All Priority Questions
# ============================================

RESULTS_PATH_ALL = "test_results_priority_tickers_MIX.json"

print("="*60)
print(f"🚀 Running {len(priority_questions)} queries...")
print("="*60)
print()

all_results = []
total = len(priority_questions)

for idx, (_, row) in enumerate(priority_questions.iterrows(), 1):
    result = await query_with_metadata(row, idx, total)
    all_results.append(result)
    
    # Save intermediate results every 10 queries
    if idx % 10 == 0:
        temp_path = RESULTS_PATH_ALL.replace('.json', f'_checkpoint_{idx}.json')
        with open(temp_path, 'w') as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)
        print(f"         💾 Checkpoint saved: {temp_path}\n")

# ============================================
# Save Final Results
# ============================================

with open(RESULTS_PATH_ALL, 'w') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("✓ All queries complete!")
print("="*60)
print()

# Summary statistics
success_count = sum(1 for r in all_results if r.get('status') == 'success')
error_count = sum(1 for r in all_results if r.get('status') == 'error')

print(f"Results saved to: {RESULTS_PATH_ALL}")
print(f"Total questions: {len(all_results)}")
print(f"  ✓ Success: {success_count}")
print(f"  ❌ Errors: {error_count}")

if success_count > 0:
    avg_context_length = np.mean([
        r['context_length'] for r in all_results 
        if r.get('status') == 'success'
    ])
    print(f"\nAverage context length: {avg_context_length:,.0f} chars")

if error_count > 0:
    print(f"\n⚠ Errors occurred in:")
    for r in all_results:
        if r.get('status') == 'error':
            print(f"  - {r['question_id']}: {r.get('error', 'Unknown')[:80]}")

print("\n" + "="*60)